In [1]:
# 1️⃣ Установка зависимостей
!pip install -q transformers accelerate torch pandas
!pip install python-docx

from docx import Document
from transformers import pipeline
import re, json
from google.colab import files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 7.8 MB/s eta 0:00:00


In [2]:
# 2️⃣ Создание pipeline
pipe = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.2", device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Device set to use cuda:0


In [9]:
import re
import json
from docx import Document
from google.colab import files

# ---------- SYSTEM PROMPT ----------
SYSTEM_PROMPT = """Ты — ассистент режиссёра.
Возвращай ТОЛЬКО JSON, начинающийся с { и заканчивающийся }.
Никаких пояснений, комментариев, ```json и текста вне JSON.
Если нет данных — пиши "".
"""

# ---------- USER PROMPT ----------
USER_PROMPT_TEMPLATE = """
Проанализируй следующую сцену и заполни строго этот JSON-шаблон:

{{
  "Серия": "{episode}",
  "НомерСцены": "",
  "Режим": "",
  "Объект": "",
  "Подобъект": "",
  "Синопсис": "",
  "Персонажи": [],
  "Массовка/Группировка": "",
  "Грим/Костюм": "",
  "Реквизит/Игровой транспорт/Животное": "",
  "Декорация": "",
  "Каскадёр/Трюк": "",
  "Администрация/Спецэффект": "",
  "Операторская техника": "",
  "Лед экраны": ""
}}

Текст сцены:
{scene_text}
"""

# ---------- READ DOCX ----------
def read_docx_text(filename):
    doc = Document(filename)
    return "\n".join(p.text for p in doc.paragraphs)

# ---------- SPLIT SCENES ----------
def normalize_text(text: str) -> str:
    """Подчищает текст перед сегментацией."""
    text = re.sub(r'\r', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def split_scenes(text: str):
    text = normalize_text(text)

    pattern = re.compile(
        r'(?=(?:^|\n)\s*'
        r'(?:СЦЕНА\s*\d+[А-ЯA-Z\-–\.]*\.?|[0-9]+[\-–\.][0-9А-ЯA-Z\-–]*\.?)\s*'
        r'(?:ИНТ|ЭКСТ|НАТ|ИНТЕРЬЕР|ЭКСТЕРЬЕР|ИНТ\.|ЭКСТ\.|НАТ\.)'
        r'.{0,120}?'
        r'(?:\s+(?:ДЕНЬ|НОЧЬ|УТРО|ВЕЧЕР|СУМЕРКИ|НОЧЬЮ|ДНЕМ|ДНЁМ))?'
        r'(?:\n|$))',
        flags=re.IGNORECASE
    )


    #добавить нахождение номера сцены, инт/экст и день/ночь/утро
    parts = re.split(pattern, text)
    scenes = [p.strip() for p in parts if p.strip()][1:]
    print(f"🧩 Найдено {len(scenes)} сцен(ы).")
    return scenes

# ---------- EXTRACT JSON ----------
def extract_json_from_output(output_text):
    """Извлекает и чинит JSON из текста модели."""
    match = re.search(r"\{.*\}", output_text, re.DOTALL)
    if not match:
        print("⚠️ JSON не найден в ответе:", output_text[:300])
        return None

    json_str = match.group(0)
    json_str = json_str.strip()
    json_str = json_str.replace("'}", "}")
    json_str = re.sub(r",\s*}", "}", json_str)
    json_str = re.sub(r",\s*\]", "]", json_str)

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print("⚠️ Ошибка JSON:", e)
        print("Фрагмент:", json_str[:400])
        return None

# ---------- SCENE ANALYSIS ----------
def extract_from_scene(scene_text, episode, pipe):
    prompt = USER_PROMPT_TEMPLATE.format(scene_text=scene_text, episode=episode)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt}
    ]

    output = pipe(messages, max_new_tokens=512)

    # 🧠 Универсальная обработка вывода (чтобы не было TypeError)
    if isinstance(output, list):
        # Если формат text-generation (обычно содержит 'generated_text')
        result = output[0].get("generated_text", output[0])
    else:
        result = output

    # Если результат — список сообщений (chat-формат)
    if isinstance(result, list):
        assistant_message = next((m.get("content", "") for m in result if m.get("role") == "assistant"), "")
        text = assistant_message
    elif isinstance(result, dict):
        text = result.get("generated_text", "")
    else:
        text = str(result)

    return extract_json_from_output(text)

# ---------- PROCESS TEXT ----------
def process_text(text, episode, pipe):
    scenes = split_scenes(text)
    results = []
    for i, scene in enumerate(scenes, start=1):
        print(f"🎬 Сцена {i}/{len(scenes)}")
        data = extract_from_scene(scene, episode, pipe)
        if data:
            results.append(data)
    return results

# ---------- MAIN ----------
print("📁 Загрузите .docx файл сценария")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
text = read_docx_text(filename)

episode_match = re.search(r"(ПЕРВАЯ|ВТОРАЯ|ТРЕТЬЯ)\s+СЕРИЯ", text, re.IGNORECASE)
episode = episode_match.group(1).capitalize() + " серия" if episode_match else "1"

results = process_text(text, episode, pipe)

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("✅ Готово! results.json создан.")

📁 Загрузите .docx файл сценария


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Saving ФИШЕР 2 сери __16.05.docx to ФИШЕР 2 сери __16.05.docx
🧩 Найдено 55 сцен(ы).
🎬 Сцена 1/55


KeyboardInterrupt: 